# Figure 1A

In [51]:
import os
import tifffile
from imaris_ims_file_reader.ims import ims
import numpy as np
import napari
from cellpose import models
from cellpose.utils import stitch3D

# import utils from the parent directory
import sys
from pathlib import Path
parent = Path.cwd().parent
if str(parent) not in sys.path:
    sys.path.insert(0, str(parent))
from utils.stitch_3d import stitch_3d 

# Load in example image to show slices
name = "ExpBSc2_TL_ITD1_20x_260513_F08"
input_directory = r"C:\Users\6331823\Local SSD\NucLogic paper figure data"
frame = 16
channel = 2 #crc channel
# image = ims(os.path.join(input_directory, f"{name}.ims"))[frame, channel] 
cropped = ims(os.path.join(input_directory, f"{name}_cropped.ims"))[frame, channel] 
print(cropped.shape)
# segmentation = tifffile.imread(os.path.join(input_directory, f"{name}_segmented.tif"))[frame, :]
# print(segmentation.shape)


# model = models.CellposeModel(
#         gpu=("cuda" == "cuda"), pretrained_model=r"C:\Users\6331823\Local SSD\20260511 CARE BSc\Train Low segmenter\models\low_input"
#     )
# model = models.CellposeModel(
#         gpu=("cuda" == "cuda"), pretrained_model=r"C:\Users\6331823\Local SSD\Coding projects\organoid_segmenter\models\2d_high_quality_model"
#     )


# cp = []
# for z in range(cropped.shape[0]):
#     print(f"Running cellpose on z-slice {z}...")
#     masks, _, _ = model.eval(cropped[z],
#                 diameter=None,
#                 normalize=True,
#                 flow_threshold=0.4,
#                 invert=False,
#                 resample=True,
#                 do_3D=False,
#                 progress=None,
#                 )
#     cp.append(masks)
# cp = np.stack(cp, axis=0)
# cp = stitch3D(cp)
# tifffile.imwrite(os.path.join(input_directory, f"{name}_cp_segmented_frame_{frame}.tif"), cp.astype(np.uint16))

# cp = tifffile.imread(os.path.join(input_directory, f"{name}_cp_segmented_frame_{frame}.tif"))
# print(f"Cellpose seg cell numbers: {len(np.unique(cp))}")
# nuclogic = stitch_3d(cp, cropped)
# nuclogic2 = stitch_3d(nuclogic, cropped)

# Show in napari
contrast = (np.median(cropped) * 1.2, np.percentile(cropped, 99.98))
scale = (2.5, 0.64, 0.64)
viewer = napari.Viewer(ndisplay=3)
viewer.dims.ndisplay = 3
# y_half = slice(0, cropped.shape[1] / 2)
# x_half = slice(0, cropped.shape[2])
viewer.add_image(cropped[:, y_half, x_half], contrast_limits=contrast, scale=scale)
# viewer.add_labels(cp[:, y_half, x_half], scale=scale, iso_gradient_mode="smooth", visible=False)
# viewer.add_labels(nuclogic[:, y_half, x_half], scale=scale, iso_gradient_mode="smooth", visible=False)
# viewer.add_image(cropped, contrast_limits=contrast, scale=scale)
# viewer.add_labels(cp, scale=scale, iso_gradient_mode="smooth", visible=False)
# viewer.add_labels(nuclogic, scale=scale, iso_gradient_mode="smooth", visible=False)
# viewer.add_labels(nuclogic2, scale=scale, iso_gradient_mode="smooth", visible=False)

# Reset view to the data bounds, then set a side-view camera angle (XZ view)
viewer.reset_view()
viewing_angle = (180, 0, 0)
viewer.camera.angles = viewing_angle

Opening readonly file: C:\Users\6331823\Local SSD\NucLogic paper figure data\ExpBSc2_TL_ITD1_20x_260513_F08_cropped.ims 

Closing file: C:\Users\6331823\Local SSD\NucLogic paper figure data\ExpBSc2_TL_ITD1_20x_260513_F08_cropped.ims 

(63, 294, 290)


In [46]:
import cv2
normalized_image = []

for z in range(cropped.shape[0]):
    slice = cropped[z]
    # distribute median to 0 and 99.99th percentile to 255
    lo = np.median(slice) * 1.2
    hi = np.percentile(slice, 99.99)

    normalized_slice = np.clip(slice, lo, hi)
    normalized_slice = ((normalized_slice - lo) / (hi - lo) * 255).astype(np.uint8)
    normalized_image.append(normalized_slice)

normalized_image = np.stack(normalized_image, axis=0)
model = models.CellposeModel(
        gpu=("cuda" == "cuda"), pretrained_model=r"C:\Users\6331823\Local SSD\Coding projects\organoid_segmenter\models\2d_high_quality_model"
    )
# mask_3d, _, _ = model.eval(normalized_image,
#     diameter=None,
#     normalize=True,
#     flow_threshold=0.4,
#     invert=False,
#     resample=True,
#     do_3D=True,
#     progress=None,
#     z_axis=0
# )
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(normalized_image, contrast_limits=(0, 255), scale=scale)
viewer.add_labels(mask_3d, scale=scale, iso_gradient_mode="smooth", visible=False)
viewer.add_labels(nuclogic, scale=scale, iso_gradient_mode="smooth", visible=False)

<Labels layer 'nuclogic' at 0x1feb3062d10>

c:\Users\6331823\AppData\Local\anaconda3\envs\organoid_segmenter\Lib\site-packages\napari\layers\labels\labels.py:908: UserWarning: Contours are not displayed during 3D rendering
  warnings.warn(
